# 04 — Team Assignment via Jersey Colour (K-Means)

Once we have stable player IDs, we need to know which team each player belongs to.  
We don't use any team-roster data — instead we cluster jersey colours.

**Approach:**
1. Crop the top half of each player's bounding box (jersey, not shorts)
2. Flatten pixels into a 2-D array `(N_pixels, 3)`
3. Run K-Means with `k=2` to find two dominant colour clusters
4. Check which cluster the *corner pixels* belong to — that's the background, not the jersey
5. The other cluster is the jersey colour → determines the team

**Connection to the pipeline:** `football_analysis/team_assigner/team_assigner.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

from football_analysis.utils.video_utils import read_video
from football_analysis.trackers.tracker import Tracker

In [ ]:
VIDEO_PATH = '../input_videos/08fd33_4.mp4'
MODEL_PATH = '../models/best.pt'
STUB_PATH  = '../stubs/track_stubs.pkl'

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame0 = cap.read()
cap.release()

# Load tracks from stub to avoid re-running YOLO
tracker = Tracker(MODEL_PATH)
tracks  = tracker.get_object_tracks([], read_from_stub=True, stub_path=STUB_PATH)
tracker.add_position_to_tracks(tracks)

print(f'Players in frame 0: {len(tracks["players"][0])}')

## Step 1 — Crop the jersey region

We use the top half of the bounding box.  
Bottom half contains shorts and often overlaps with pitch grass, which confuses clustering.

In [ ]:
def get_jersey_crop(frame, bbox):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    mid_y = (y1 + y2) // 2
    return frame[y1:mid_y, x1:x2]

# Show jersey crops for the first 6 detected players
player_ids = list(tracks['players'][0].keys())[:6]
fig, axes  = plt.subplots(1, len(player_ids), figsize=(14, 3))

for ax, pid in zip(axes, player_ids):
    bbox = tracks['players'][0][pid]['bbox']
    crop = get_jersey_crop(frame0, bbox)
    ax.imshow(crop[:, :, ::-1])
    ax.set_title(f'ID {pid}')
    ax.axis('off')

plt.suptitle('Jersey crops (top half of bbox)', fontsize=12)
plt.tight_layout()
plt.show()

## Step 2 — K-Means clustering on one crop

We reshape the crop from `(H, W, 3)` to `(H*W, 3)` — each row is one pixel in BGR space.  
K-Means finds 2 cluster centres: one for the jersey, one for everything else (grass, sky, etc.).

In [ ]:
pid  = player_ids[0]
bbox = tracks['players'][0][pid]['bbox']
crop = get_jersey_crop(frame0, bbox)

pixels = crop.reshape(-1, 3).astype(np.float32)
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(pixels)

print('Cluster centres (BGR):')
for i, c in enumerate(kmeans.cluster_centers_):
    print(f'  Cluster {i}: B={c[0]:.0f}  G={c[1]:.0f}  R={c[2]:.0f}')

print()
print('Labels (0 or 1 per pixel):', kmeans.labels_[:20], '...')

## Step 3 — Which cluster is the background?

The corners of the bounding box are almost certainly background (grass).  
We check which cluster the corner pixels belong to — that cluster is the background.  
The *other* cluster is the jersey.

In [ ]:
labels_2d = kmeans.labels_.reshape(crop.shape[:2])

# Corner pixels
corners = [
    labels_2d[0, 0], labels_2d[0, -1],
    labels_2d[-1, 0], labels_2d[-1, -1],
]
background_cluster = max(set(corners), key=corners.count)
jersey_cluster     = 1 - background_cluster

jersey_colour_bgr = kmeans.cluster_centers_[jersey_cluster]
jersey_colour_rgb = jersey_colour_bgr[::-1].astype(int)

print(f'Background cluster: {background_cluster}')
print(f'Jersey cluster    : {jersey_cluster}')
print(f'Jersey colour (RGB): {jersey_colour_rgb}')

# Show the segmentation
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(crop[:, :, ::-1])
axes[0].set_title('Original crop')
axes[0].axis('off')

axes[1].imshow(labels_2d, cmap='RdYlBu')
axes[1].set_title('K-Means labels (2 clusters)')
axes[1].axis('off')

jersey_mask = (labels_2d == jersey_cluster).astype(np.uint8) * 255
axes[2].imshow(jersey_mask, cmap='gray')
axes[2].set_title('Jersey mask')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Step 4 — Assign teams across all players

We run K-Means once on frame 0 to find the **two team colours** (the centroids of all jersey colours across all players).  
For subsequent frames, each player's jersey colour is compared to those two centroids — nearest wins.

In [ ]:
from football_analysis.team_assigner.team_assigner import TeamAssigner

assigner = TeamAssigner()
assigner.assign_team_color(frame0, tracks['players'][0])

print('Team 1 colour (BGR):', assigner.team_colors[1])
print('Team 2 colour (BGR):', assigner.team_colors[2])

# Show swatches
fig, axes = plt.subplots(1, 2, figsize=(4, 2))
for i, team_id in enumerate([1, 2]):
    bgr = np.array(assigner.team_colors[team_id], dtype=np.uint8)
    rgb = bgr[::-1]
    swatch = np.full((50, 100, 3), rgb, dtype=np.uint8)
    axes[i].imshow(swatch)
    axes[i].set_title(f'Team {team_id}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Visualise team labels on the pitch

In [ ]:
annotated = frame0.copy()

for pid, info in tracks['players'][0].items():
    team = assigner.get_player_team(frame0, info['bbox'], pid)
    colour = assigner.team_colors[team]  # BGR

    x1, y1, x2, y2 = [int(v) for v in info['bbox']]
    cx = (x1 + x2) // 2
    w  = x2 - x1
    cv2.ellipse(annotated, (cx, y2), (w//2, int(w*0.35)), 0, -45, 235, colour, 3)

plt.figure(figsize=(16, 9))
plt.imshow(annotated[:, :, ::-1])
plt.title('Players coloured by team (K-Means jersey clustering)')
plt.axis('off')
plt.show()

## Why cache assignments per player ID?

K-Means on every frame would be slow and flicker (jersey in shadow vs. sun looks different).  
The `TeamAssigner` caches the team assignment once per player ID and reuses it — this gives stable colours throughout the video.

## Key takeaways

| Concept | Detail |
|---|---|
| K-Means `k=2` | Separates jersey pixels from background in each crop |
| Corner trick | Corners ≈ background — identifies which cluster to discard |
| Team-level K-Means | A second K-Means over all players' jersey colours finds the two team colours |
| Caching | Assignment stored per track ID to avoid flicker across frames |

**Next:** `05_camera_movement.ipynb` — compensating for the camera panning and tilting